# Cobertura y Conectividad Móvil en México
## Documentación ETL
### Datasets: Definición y uso
1. **`master_municipal`:**
   - **Definición:** Dataset agregado a nivel municipal.
   - **Uso:** Contiene métricas como velocidad promedio, porcentaje de cobertura relativa por tecnología (3G, 4G, 5G) y la tecnología predominante en cada municipio.
   - **Exportación:** Archivo CSV (`master_municipal_2026.csv`).

2. **`master_estatal`:**
   - **Definición:** Dataset agregado a nivel estatal.
   - **Uso:** Contiene métricas similares a `master_municipal`, pero agregadas a nivel estatal.
   - **Exportación:** Archivo CSV (`master_estatal_2026.csv`).

3. **`mexico_granular`:**
   - **Definición:** Dataset granular con geometrías hexagonales.
   - **Uso:** Contiene datos detallados de cada hexágono, como velocidad promedio y tecnología predominante.
   - **Exportación:** Archivo GeoJSON (`mexico_granular_2026.geojson`).

### Fuentes
1. **Ookla:**
   - **Descripción:** Datos de velocidad y conectividad móvil.

2. **Marco Geoestadístico del INEGI:**
   - **Descripción:** Límites municipales y estatales de México.

### Diccionario: Variables creadas
- **`avg_d_mbps`:** Velocidad promedio de descarga en Mbps.
- **`tech_proxy`:** Tecnología predominante en el área (3G, 4G, 5G).
- **`int_pct_3g_coverage`:** Porcentaje promedio de cobertura 3G.
- **`int_pct_4g_coverage`:** Porcentaje promedio de cobertura 4G.
- **`int_pct_5g_coverage`:** Porcentaje promedio de cobertura 5G.
- **`pct_cobertura_absoluta`:** Porcentaje de cobertura total en el municipio basado en el área cubierta por hexágonos (pendiente de implementación).
- **`tests`:** Número total de pruebas realizadas en el área.

### Raw: Archivos originales
  - `00mun.shp`: Límites municipales.
  - `00est.shp`: Límites estatales.
  - `gps_mobile_tiles.shp`: Hexágonos de conectividad móvil.

### Proxy: Metodología para determinar la categoría
- **Definición:** La categoría de tecnología (`tech_proxy`) se determina en función de la velocidad promedio de descarga:
  - **3G:** Velocidad menor a 10 Mbps.
  - **4G:** Velocidad entre 10 y 40 Mbps.
  - **5G:** Velocidad mayor a 40 Mbps.
- **Justificación:** Este proxy se basa en estándares generales (usados por el IFT para sus propios mapas) de capacidad de red para cada tecnología.

### Filtro de confianza
- **Definición:** Se filtran los hexágonos con menos de `n` pruebas realizadas.
- **Justificación:** Este filtro asegura que los datos utilizados sean representativos y confiables, eliminando áreas con poca información que podrían sesgar los resultados.


In [1]:
import geopandas as gpd
import pandas as pd
import os

## Carga de shapefiles

In [ ]:
path = os.path.join('..', 'data', 'raw')

# Verificar si los archivos existen
municipios_path = os.path.join(path, '00mun.shp')
ookla_path = os.path.join(path, 'gps_mobile_tiles.shp')

municipios = gpd.read_file(municipios_path)
ookla_global = gpd.read_file(ookla_path)

# Corroborar CRS
if municipios.crs != "EPSG:3857":
    municipios = municipios.to_crs("EPSG:3857")
ookla_global = ookla_global.to_crs("EPSG:3857")

### CRS: Proyecciones usadas
- **Proyección inicial:** Los datos de los shapefiles (`00mun.shp`, `00est.shp` y `gps_mobile_tiles.shp`) se cargan con sus sistemas de referencia espacial originales.
- **Proyección utilizada:** Se reproyectan al sistema de coordenadas EPSG:3857 (Web Mercator) para realizar operaciones espaciales como el `Spatial Join`. Finalmente, los datos se transforman a EPSG:4326 (WGS 84) para exportar los resultados en un formato estándar compatible con herramientas de visualización y análisis.

In [5]:
ookla_global.shape

(3311092, 7)

## Filtrado México

In [31]:
# Bounding Box México
minx, miny, maxx, maxy = -118.5, 14.5, -86.5, 32.7
ookla_mexico = ookla_global.cx[minx:maxx, miny:maxy].copy()

ookla_mexico.shape

(103505, 7)

In [32]:
# Spatial Join (usando centroide)
ookla_mexico['geometry_backup'] = ookla_mexico.geometry
ookla_mexico['geometry'] = ookla_mexico.geometry.centroid

# Cruce con municipios
ookla_final = gpd.sjoin(ookla_mexico, municipios, how="inner", predicate="within")

# Restauración de la geometría de hexágono para el dataset granular
ookla_final['geometry'] = ookla_final['geometry_backup']
ookla_final = ookla_final.to_crs("EPSG:4326")

## Limpieza e intervalos (inferencia de tecnología)

In [ ]:
# Filtro de confianza
n = 
3ookla_final = ookla_final[ookla_final['tests'] >= n].copy()
ookla_final['avg_d_mbps'] = ookla_final['avg_d_kbps'] / 1000

# Clasificación por tecnología (Proxy de capacidad de red)
def clasificar_tech(speed):
    if speed < 10: return '3G'
    if speed < 40: return '4G'
    return '5G'

ookla_final['tech_proxy'] = ookla_final['avg_d_mbps'].apply(clasificar_tech)

## Generación de datasets (municipal y estatal)

In [34]:
len(ookla_final["NOMGEO"].unique())

1401

In [39]:
# Calcular porcentajes de cobertura para cada tecnología
ookla_final['int_pct_3g_coverage'] = ookla_final['tech_proxy'].apply(lambda x: 100 if x == '3G' else 0)
ookla_final['int_pct_4g_coverage'] = ookla_final['tech_proxy'].apply(lambda x: 100 if x == '4G' else 0)
ookla_final['int_pct_5g_coverage'] = ookla_final['tech_proxy'].apply(lambda x: 100 if x == '5G' else 0)

# Agrupar por municipio
master_municipal = ookla_final.groupby(['CVE_ENT', 'CVEGEO', 'NOMGEO']).agg({
    'avg_d_mbps': 'mean',  # Velocidad promedio
    'tech_proxy': lambda x: x.mode()[0],  # Tecnología predominante
    'geometry': 'count',  # Placeholder para calcular porcentajes
    'tests': 'sum',  # Total de tests
    'int_pct_3g_coverage': 'mean',  # Porcentaje promedio de cobertura 3G
    'int_pct_4g_coverage': 'mean',  # Porcentaje promedio de cobertura 4G
    'int_pct_5g_coverage': 'mean'   # Porcentaje promedio de cobertura 5G
}).reset_index()

# Renombrar columnas para cumplir con los requisitos
master_municipal.rename(columns={
    'CVEGEO': 'id_cvegeo',
    'NOMGEO': 'nom_mun',
    'avg_d_mbps': 'int_avg_speed',
    'tech_proxy': 'int_tech_predominante',
    'int_pct_3g_coverage': 'int_pct_3g_coverage',
    'int_pct_4g_coverage': 'int_pct_4g_coverage',
    'int_pct_5g_coverage': 'int_pct_5g_coverage'
}, inplace=True)

# Dataset Estatal
master_estatal = master_municipal.groupby(['CVE_ENT']).agg({
    'int_avg_speed': 'mean',
    'int_pct_3g_coverage': 'mean',
    'int_pct_4g_coverage': 'mean',
    'int_pct_5g_coverage': 'mean',
    'int_tech_predominante': lambda x: x.mode()[0]  # Tecnología predominante
}).reset_index()

In [42]:
ookla_final.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,geometry,geometry_backup,index_right,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO,avg_d_mbps,tech_proxy,int_pct_3g_coverage,int_pct_4g_coverage,int_pct_5g_coverage,area_hex
111114,0230132213232332,21231,10613,143,3,1,"POLYGON ((-116.94946 32.55144, -116.94397 32.5...","POLYGON ((-116.94946 32.55144, -116.94397 32.5...",13,02004,02,004,Tijuana,21.231,4G,0,100,0,0.000025
111124,0230132213233222,27581,15086,57,13,8,"POLYGON ((-116.93848 32.55144, -116.93298 32.5...","POLYGON ((-116.93848 32.55144, -116.93298 32.5...",13,02004,02,004,Tijuana,27.581,4G,0,100,0,0.000025
111125,0230132213233223,114460,6763,78,13,3,"POLYGON ((-116.93298 32.55144, -116.92749 32.5...","POLYGON ((-116.93298 32.55144, -116.92749 32.5...",13,02004,02,004,Tijuana,114.460,5G,0,0,100,0.000025
111126,0230132213233232,33483,4855,81,23,4,"POLYGON ((-116.92749 32.55144, -116.922 32.551...","POLYGON ((-116.92749 32.55144, -116.922 32.551...",13,02004,02,004,Tijuana,33.483,4G,0,100,0,0.000025
111128,0230132213233322,50302,14419,78,14,5,"POLYGON ((-116.9165 32.55144, -116.91101 32.55...","POLYGON ((-116.9165 32.55144, -116.91101 32.55...",13,02004,02,004,Tijuana,50.302,5G,0,0,100,0.000025


In [48]:
print("Valores únicos en municipios['CVEGEO']:", len(municipios['CVEGEO'].unique()))
print("Valores únicos en ookla_final['CVEGEO']:", len(ookla_final['CVEGEO'].unique()))

Valores únicos en municipios['CVEGEO']: 2478
Valores únicos en ookla_final['CVEGEO']: 1481


### Añadir porcentaje de cobertura absoluto (territorial) -- PENDIENTE

In [49]:
municipios.head()

,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO,geometry,area_total
0,01001,01,001,Aguascalientes,"POLYGON ((-102.10731 22.07475, -102.10698 22.0...",0.102461
1,01008,01,008,San José de Gracia,"POLYGON ((-102.45536 22.31212, -102.455 22.312...",0.070078
2,01011,01,011,San Francisco de los Romo,"POLYGON ((-102.15937 22.099, -102.15637 22.097...",0.012176
3,01009,01,009,Tepezalá,"POLYGON ((-102.17737 22.36243, -102.17967 22.3...",0.020329
4,01007,01,007,Rincón de Romos,"POLYGON ((-102.22683 22.37392, -102.2266 22.37...",0.032932


In [ ]:
# Calcular el área total de cada municipio
municipios['area_total'] = municipios.geometry.area

# Calcular el área cubierta por hexágonos en cada municipio
ookla_final['area_hex'] = ookla_final.geometry.area

# Agrupar por municipio para sumar el área cubierta
area_cobertura = ookla_final.groupby(['CVE_ENT', 'CVEGEO', 'NOMGEO']).agg({
    'area_hex': 'sum'
}).reset_index()

# Combinar con el área total de los municipios (asegurando que todos los municipios estén presentes)
area_cobertura = municipios[['CVEGEO', 'area_total']].merge(
    area_cobertura, on='CVEGEO', how='left'
)

# Rellenar valores faltantes para municipios sin cobertura
area_cobertura['area_hex'] = area_cobertura['area_hex'].fillna(0)

# Calcular el porcentaje de cobertura absoluto
area_cobertura['pct_cobertura_absoluta'] = (area_cobertura['area_hex'] / area_cobertura['area_total']) * 100

# Combinar con el dataset municipal
master_municipal = master_municipal.merge(
    area_cobertura[['CVEGEO', 'pct_cobertura_absoluta']], on='CVEGEO', how='left'
)

KeyError: 'CVEGEO'

## Exportación

In [54]:
master_municipal.to_csv('../data/processed/master_municipal_2026.csv', index=False)
master_estatal.to_csv('../data/processed/master_estatal_2026.csv', index=False)
# El dataset granular para mapas (solo México)
ookla_final[['tech_proxy', 'avg_d_mbps', 'geometry']].to_file('../data/processed/mexico_granular_2026.geojson', driver='GeoJSON')

print("Datasets exportados.")

Datasets exportados.
